Token
13342~2ztTUmtULKck7M3PvmrQ9BCvNzTJ9z7KkTWNMu783K8MJNJEDaKE4B68Ru6Z4RuB
Copy this token down now. Once you leave this page you won't be able to retrieve the full token anymore, you'll have to regenerate it to get a new value.
App
User-Generated
Purpose
Download_Files
Created
Feb 28 at 3:29pm
Last Used
--
Expires
Mar 31 at 12am


In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env into os.environ

In [ ]:
CANVAS_DOMAIN = "https://rady.instructure.com/"
ACCESS_TOKEN = os.getenv("CANVAS_ACCESS_TOKEN")

if not ACCESS_TOKEN:
    raise RuntimeError("CANVAS_ACCESS_TOKEN not set in .env file")


def get_courses():
    url = f"{CANVAS_DOMAIN}/api/v1/users/self/courses"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

    params = {
        "enrollment_state[]": [
            "active",
            "completed",
            "deleted",
        ],
        "per_page": 30,
    }

    all_courses = []
    page = 1

    while True:
        params["page"] = page
        response = requests.get(url, headers=headers, params=params)

        if response.status_code != 200:
            print(f"Error fetching courses: {response.status_code} - {response.text}")
            break

        courses = response.json()
        if not courses:
            break

        all_courses.extend(courses)
        page += 1

    return all_courses


courses = get_courses()

course_dict = {
    course.get("id"): course.get("original_name")
    or course.get("name")
    or "(no name field)"
    for course in courses
}

for k, v in course_dict.items():
    print(f"{k}: {v}")

In [27]:
ROOT_DIR = "./Courses"  # Root directory for all courses


def create_directory(path):
    if not os.path.exists(path):
        os.makedirs(path)


def download_file(file_url, file_path):
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    r = requests.get(file_url, headers=headers, stream=True)
    r.raise_for_status()

    with open(file_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


In [25]:
import requests


def get_files(course_id, course_name):
    url = f"{CANVAS_DOMAIN}/api/v1/courses/{course_id}/files"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

    response = requests.get(url, headers=headers)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        # include Canvas error body (often helpful)
        print(
            f"Error fetching files for course {course_id}: {response.status_code} - {response.text[:300]}"
        )
        raise

    files = response.json()
    files_dir = os.path.join(ROOT_DIR, course_name, "Files")
    create_directory(files_dir)

    for f in files:
        file_path = os.path.join(files_dir, f["filename"])
        download_file(f["url"], file_path)


In [26]:
def get_modules(course_id, course_name):
    url = f"{CANVAS_DOMAIN}/api/v1/courses/{course_id}/modules"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Error fetching modules for course {course_id}: {response.status_code}")
        return

    modules = response.json()
    modules_dir = os.path.join(ROOT_DIR, course_name, "Modules")
    create_directory(modules_dir)

    for module in modules:
        module_items_url = (
            f"{CANVAS_DOMAIN}/api/v1/courses/{course_id}/modules/{module['id']}/items"
        )
        items_response = requests.get(module_items_url, headers=headers)

        if items_response.status_code != 200:
            print(
                f"Error fetching items for module {module['id']}: {items_response.status_code}"
            )
            continue

        items = items_response.json()
        for item in items:
            if item.get("type") == "File":
                file_id = item.get("content_id")
                if not file_id:
                    print(f"Skipping file item with no content_id: {item.get('title')}")
                    continue

                # 1) Look up file metadata to get a real download URL
                file_meta_url = f"{CANVAS_DOMAIN}/api/v1/files/{file_id}"
                meta_resp = requests.get(file_meta_url, headers=headers)
                if meta_resp.status_code != 200:
                    print(
                        f"Error fetching file metadata {file_id}: {meta_resp.status_code}"
                    )
                    continue

                meta = meta_resp.json()

                # Canvas commonly provides 'url' or 'download_url' depending on instance/settings
                file_url = meta.get("url") or meta.get("download_url")
                if not file_url:
                    print(
                        f"No downloadable URL for file {file_id} ({item.get('title')})"
                    )
                    continue

                file_path = os.path.join(modules_dir, item.get("title") or f"{file_id}")
                download_file(file_url, file_path)


In [16]:
def get_assignments(course_id, course_name):
    url = f"{CANVAS_DOMAIN}/api/v1/courses/{course_id}/assignments"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        assignments = response.json()
        assignments_dir = os.path.join(ROOT_DIR, course_name, "Assignments")
        create_directory(assignments_dir)

        for assignment in assignments:
            if assignment["submission_types"]:
                for submission_type in assignment["submission_types"]:
                    if submission_type == "online_upload":
                        # Here you could add code to download uploaded files if available
                        print(f"Assignment {assignment['name']} allows file uploads.")
    else:
        print(
            f"Error fetching assignments for course {course_id}: {response.status_code}"
        )


In [28]:
create_directory(ROOT_DIR)

import os
import traceback

try:
    create_directory(ROOT_DIR)
except Exception as e:
    print(f"[FATAL] Failed to create root directory: {ROOT_DIR}")
    print(f"Error: {e}")
    traceback.print_exc()

for course_id, course_name in course_dict.items():
    print(f"\nProcessing course: {course_name} (ID: {course_id})")

    try:
        course_path = os.path.join(ROOT_DIR, course_name)
        create_directory(course_path)
        print("  ✓ Directory created")
    except Exception as e:
        print(f"  ✗ Failed to create directory for {course_name}")
        print(f"    Error: {e}")
        traceback.print_exc()
        continue  # skip to next course

    try:
        get_files(course_id, course_name)
        print("  ✓ Files retrieved")
    except Exception as e:
        print(f"  ✗ get_files() failed for {course_name}")
        print(f"    Error: {e}")
        traceback.print_exc()

    try:
        get_modules(course_id, course_name)
        print("  ✓ Modules retrieved")
    except Exception as e:
        print(f"  ✗ get_modules() failed for {course_name}")
        print(f"    Error: {e}")
        traceback.print_exc()

    try:
        get_assignments(course_id, course_name)
        print("  ✓ Assignments retrieved")
    except Exception as e:
        print(f"  ✗ get_assignments() failed for {course_name}")
        print(f"    Error: {e}")
        traceback.print_exc()



Processing course: MGTA403-907772-907773: AI-Assisted Math & Programming for Business Analytics (Nijs) -- MS26 (ID: 1801)
  ✓ Directory created
Error fetching files for course 1801: 403 - {"status":"unauthorized","errors":[{"message":"user not authorized to perform that action"}]}
  ✗ get_files() failed for MGTA403-907772-907773: AI-Assisted Math & Programming for Business Analytics (Nijs) -- MS26
    Error: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1801/files


Traceback (most recent call last):
  File "/tmp/ipykernel_45477/4009183818.py", line 27, in <module>
    get_files(course_id, course_name)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_45477/2438732583.py", line 10, in get_files
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/base-uv/.venv/lib/python3.13/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1801/files


  ✓ Modules retrieved
  ✓ Assignments retrieved

Processing course: MGTA444-999022: Business Analytics Consulting (Peterson) -- MS26 (ID: 1955)
  ✓ Directory created
Error fetching files for course 1955: 403 - {"status":"unauthorized","errors":[{"message":"user not authorized to perform that action"}]}
  ✗ get_files() failed for MGTA444-999022: Business Analytics Consulting (Peterson) -- MS26
    Error: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1955/files


Traceback (most recent call last):
  File "/tmp/ipykernel_45477/4009183818.py", line 27, in <module>
    get_files(course_id, course_name)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_45477/2438732583.py", line 10, in get_files
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/base-uv/.venv/lib/python3.13/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1955/files


  ✓ Modules retrieved
Assignment CLASS 1 allows file uploads.
Assignment CLASS 2 allows file uploads.
Assignment CLASS 4 allows file uploads.
Assignment CLASS 5 allows file uploads.
Assignment HWK Practice Capstone Proposal allows file uploads.
Assignment Class 3 HWK: Case Review allows file uploads.
Assignment NEW SUBMISSION PAGE- CLASSWORK 2 HOMEWORK allows file uploads.
  ✓ Assignments retrieved

Processing course: MGTA451-907770: Business Analytics in Marketing, Finance, and Ops (Buti / Shin / Wilbur) -- MS26 (ID: 1804)
  ✓ Directory created
Error fetching files for course 1804: 403 - {"status":"unauthorized","errors":[{"message":"user not authorized to perform that action"}]}
  ✗ get_files() failed for MGTA451-907770: Business Analytics in Marketing, Finance, and Ops (Buti / Shin / Wilbur) -- MS26
    Error: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1804/files


...

  ✓ Modules retrieved
  ✓ Assignments retrieved

Processing course: MSBA Program 2026 (ID: 1815)
  ✓ Directory created
Error fetching files for course 1815: 403 - {"status":"unauthorized","errors":[{"message":"user not authorized to perform that action"}]}
  ✗ get_files() failed for MSBA Program 2026
    Error: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1815/files


Traceback (most recent call last):
  File "/tmp/ipykernel_45477/4009183818.py", line 27, in <module>
    get_files(course_id, course_name)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_45477/2438732583.py", line 10, in get_files
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/base-uv/.venv/lib/python3.13/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1815/files


  ✓ Modules retrieved
Assignment Complete all assigned Tracks, Courses, and Chapters on DataCamp allows file uploads.
Assignment Complete the Math and Stats Bootcamp allows file uploads.
  ✓ Assignments retrieved

Processing course: RadyReady 2025 (ID: 1816)
  ✓ Directory created
Error fetching files for course 1816: 403 - {"status":"unauthorized","errors":[{"message":"user not authorized to perform that action"}]}
  ✗ get_files() failed for RadyReady 2025
    Error: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1816/files


Traceback (most recent call last):
  File "/tmp/ipykernel_45477/4009183818.py", line 27, in <module>
    get_files(course_id, course_name)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_45477/2438732583.py", line 10, in get_files
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/base-uv/.venv/lib/python3.13/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 403 Client Error: Forbidden for url: https://rady.instructure.com//api/v1/courses/1816/files


  ✓ Modules retrieved
  ✓ Assignments retrieved
